# REEFPRINT (aka KHANYA) — S2 v2 training

Private research notebook template for Mintek Problem 3. It uses a hash-verified snapshot of the DeepLabV3–ResNet50 training and held-out evaluation modules from commit `ac7c058e9a52b9ea1004925085e38071e2892b16` on the `codex/khanya-build-plan` branch.

The user has authorized a **private** Kaggle training run. The notebook downloads the source archive at runtime and does not create a Kaggle dataset mirror. Keep the Kaggle dataset and notebook private; this authorization is not a legal determination of commercial redistribution rights. This notebook will not publish the dataset. Never attach S1, S3, V1, an unrelated public mirror, or generated data as if it were S2 v2.

In notebook Settings enable a GPU and Internet (the repo clone and initial Torchvision pretrained weights need network). The private notebook fetches the official S2 v2 ZIP from the publisher over Kaggle Internet at run time; no Kaggle dataset mirror is needed. At startup this notebook safely extracts the archive and verifies 37 train and 12 test image/mask pairs, prints the assigned accelerator and stops if inputs are missing or ambiguous. It runs the current default training configuration (8 epochs; 64 train patches/epoch; 32 validation patches; 512-pixel patches). It leaves the 12-image test split for frozen evaluation, then packages only the checkpoint, report, logs and run manifest under `/kaggle/working` for private download. Never publish outputs.


In [ ]:
from pathlib import Path, PurePosixPath
from urllib.parse import quote
from urllib.request import urlopen
from zipfile import ZipFile
import hashlib, json, uuid

publisher_share = 'https://disk.360.yandex.ru/d/wYK_5JyQy0pIcg'
api = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key=' + quote(publisher_share, safe='')
archive = Path('/kaggle/working/S2_v2.zip')
expected_sha256 = '64aebd108b3306a8b2e83a945c01f2779bbfd1abeb187b1e2cedcac96cf40c1d'
with urlopen(api, timeout=60) as response:
    download_url = json.load(response)['href']
sha = hashlib.sha256()
size = 0
with urlopen(download_url, timeout=180) as source, archive.open('wb') as target:
    while True:
        chunk = source.read(8 * 1024 * 1024)
        if not chunk:
            break
        target.write(chunk)
        sha.update(chunk)
        size += len(chunk)
        if size % (64 * 1024 * 1024) < len(chunk):
            print(f'Downloaded {size // (1024*1024)} MiB', flush=True)
archive_sha256 = sha.hexdigest()
if archive_sha256 != expected_sha256:
    raise RuntimeError(f'Publisher archive SHA-256 mismatch: {archive_sha256}')
print('Publisher ZIP bytes:', size, '| SHA-256:', archive_sha256)

extract_base = Path('/kaggle/working') / f's2-v2-source-{uuid.uuid4().hex[:8]}'
extract_base.mkdir(parents=True, exist_ok=False)
with ZipFile(archive) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f'Archive CRC check failed at {bad}')
    for member in zf.infolist():
        path = PurePosixPath(member.filename)
        if path.is_absolute() or '..' in path.parts:
            raise RuntimeError(f'Refusing unsafe archive path: {member.filename}')
    zf.extractall(extract_base)

roots = []
for train_dir in extract_base.rglob('imgs/train'):
    candidate = train_dir.parent.parent
    if (candidate / 'imgs/test').is_dir() and (candidate / 'masks/train').is_dir() and (candidate / 'masks/test').is_dir():
        train_imgs = {p.stem for p in (candidate / 'imgs/train').glob('*.jpg')}
        test_imgs = {p.stem for p in (candidate / 'imgs/test').glob('*.jpg')}
        train_masks = {p.stem for p in (candidate / 'masks/train').glob('*.png')}
        test_masks = {p.stem for p in (candidate / 'masks/test').glob('*.png')}
        if len(train_imgs) == 37 and len(test_imgs) == 12 and train_imgs == train_masks and test_imgs == test_masks:
            roots.append(candidate)
if len(roots) != 1:
    raise RuntimeError(f'Expected exactly one S2-v2 root with 37/12 paired image-mask files; found {len(roots)}. Stop; do not proceed with a different subset.')
s2_root = roots[0].resolve()
print('Verified S2 v2 path:', s2_root)
print('Train pairs:', 37, '| held-out test pairs:', 12)
print('No test images will be used for training or checkpoint selection.')


In [ ]:
import base64, hashlib, io, json, time, zipfile
from pathlib import Path

branch = 'codex/khanya-build-plan'
git_sha = 'ac7c058e9a52b9ea1004925085e38071e2892b16'
source_bundle_sha256 = '1bf02ae874ca1da61f84e05d7a86d4fd3254b93ae2bea2267c6e8ced31f4c775'
source_bundle_b64 = """..."""
run_id = time.strftime('%Y%m%d-%H%M%S')
repo = Path('/kaggle/working') / f'khanya-{run_id}'
repo.mkdir(parents=True, exist_ok=False)
bundle = base64.b64decode(source_bundle_b64, validate=True)
if hashlib.sha256(bundle).hexdigest() != source_bundle_sha256:
    raise RuntimeError('Embedded source bundle SHA-256 mismatch; stop without training.')
with zipfile.ZipFile(io.BytesIO(bundle)) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f'Embedded source ZIP CRC check failed at {bad}')
    for member in zf.infolist():
        if member.filename.startswith('/') or '..' in Path(member.filename).parts:
            raise RuntimeError(f'Unsafe embedded source path: {member.filename}')
    zf.extractall(repo)
source_hashes = {}
for path in sorted((repo / 'src/segmentation').glob('*.py')):
    source_hashes[str(path.relative_to(repo)).replace('\\', '/')] = hashlib.sha256(path.read_bytes()).hexdigest()
print('Run ID:', run_id, '| source branch:', branch, '| commit:', git_sha)
print('Embedded source bundle SHA-256:', source_bundle_sha256)
print('Training source modules:', len(source_hashes))
print('Materialized code root:', repo)
data_link = repo / 'data/raw/lumenstone/S2_v2'
data_link.parent.mkdir(parents=True, exist_ok=True)
data_link.symlink_to(s2_root, target_is_directory=True)
train_path = data_link / 'imgs/train'
test_path = data_link / 'imgs/test'
if len(list(train_path.glob('*.jpg'))) != 37 or len(list(test_path.glob('*.jpg'))) != 12:
    raise RuntimeError(f'S2 link is not readable at trainer path: {data_link}')
print('Verified trainer data link:', data_link, '->', s2_root)


In [ ]:
import os, sys, torch, torchvision

os.chdir(repo)
os.environ['KHANYA_SUBSET'] = 'S2'
print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. Enable GPU in Kaggle notebook settings, reconnect, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f'GPU memory free/total: {free_bytes/2**30:.1f}/{total_bytes/2**30:.1f} GiB')
print('Default training budget: 8 epochs, 64 train patches/epoch, 32 validation patches, 512px patches, batch 2, AdamW lr 2e-4.')


## Train once, then freeze the candidate

The validation split is carved from the 37 training images. Inspect its trend and the saved run log. Do not use the held-out 12 test images to tune epochs, thresholds, or class selection. The first epoch includes download/cache overhead for the declared pretrained weights; the time shown below is the measured Kaggle run, not a guaranteed duration. The default run is the controlled baseline; only change training settings based on validation evidence, and record every change.


In [ ]:
import subprocess, sys
from pathlib import Path

ckpt_dir = repo / 'checkpoints/lumenstone_s2_patches'
if ckpt_dir.exists():
    raise RuntimeError(f'Checkpoint directory already exists at {ckpt_dir}. Start a fresh notebook session/run rather than accidentally resuming or overwriting artifacts.')
log_path = Path('/kaggle/working') / f'khanya-s2-train-{run_id}.log'
with log_path.open('w') as log_file:
    result = subprocess.run([sys.executable, '-m', 'src.segmentation.train_patches'],
                            cwd=repo, stdout=log_file, stderr=subprocess.STDOUT, text=True)
print(log_path.read_text(errors='replace')[-12000:])
if result.returncode != 0 or not (ckpt_dir / 'best.pt').is_file():
    raise RuntimeError('Training did not produce best.pt. Preserve the log and fix the stated failure before evaluation.')
print('Candidate checkpoint:', ckpt_dir / 'best.pt')


## Frozen held-out evaluation

This evaluates the selected checkpoint on the 12 publisher test images. It does not reselect the checkpoint. Report all five classes including background and magnetite; the historical magnetite failure must not be hidden. Full-resolution sliding-window evaluation can take additional time.


In [ ]:
import subprocess, sys
from pathlib import Path

eval_log = Path('/kaggle/working') / f'khanya-s2-eval-{run_id}.log'
with eval_log.open('w') as log_file:
    result = subprocess.run([sys.executable, '-m', 'src.segmentation.train_patches', '--eval'],
                            cwd=repo, stdout=log_file, stderr=subprocess.STDOUT, text=True)
print(eval_log.read_text(errors='replace')[-16000:])
report_path = repo / 'reports/lumenstone_s2_patches_test_metrics.json'
if result.returncode != 0 or not report_path.is_file():
    raise RuntimeError('Held-out evaluation/report failed. Keep the training artifact and troubleshoot; do not present validation results as test accuracy.')
metrics = json.loads(report_path.read_text())
print('Class names:', metrics.get('class_names'))
print('Per-class IoU:', metrics.get('iou_per_class'))
print('Pooled mIoU:', metrics.get('mean_iou'))
print('Pixel accuracy:', metrics.get('pixel_accuracy'))


In [ ]:
import importlib.metadata, hashlib, zipfile
from pathlib import Path

checkpoint = ckpt_dir / 'best.pt'
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
manifest = {
    'run_id': run_id,
    'git_commit': git_sha,
    'code_delivery': 'hash-verified embedded training-module snapshot',
    'source_bundle_sha256': source_bundle_sha256,
    'source_file_sha256': source_hashes,
    'dataset': 'LumenStone S2 v2',
    'dataset_archive_sha256': archive_sha256,
    'dataset_archive_bytes': size,
    'dataset_source': 'https://imaging.cs.msu.ru/en/research/geology/lumenstone',
    'train_pairs': 37,
    'validation_pairs': 6,
    'test_pairs': 12,
    'test_used_for_selection': False,
    'subset': 'S2',
    'training_budget': {'epochs': 8, 'patches_per_epoch': 64, 'validation_patches': 32, 'patch_px': 512, 'batch_size': 2, 'learning_rate': 0.0002},
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'checkpoint_sha256': digest,
    'test_metrics': metrics,
    'rights_note': 'Private research run. Do not publish dataset or derived checkpoint without rights review.'
}
manifest_path = Path('/kaggle/working') / f'khanya-s2-manifest-{run_id}.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
bundle_path = Path('/kaggle/working') / f'khanya-s2-run-{run_id}.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
    archive.write(checkpoint, 'checkpoints/lumenstone_s2_patches/best.pt')
    archive.write(report_path, 'reports/lumenstone_s2_patches_test_metrics.json')
    archive.write(log_path, 'logs/training.log')
    archive.write(eval_log, 'logs/evaluation.log')
    archive.write(manifest_path, 'run_manifest.json')
print('Checkpoint SHA256:', digest)
print('Private run bundle:', bundle_path, f'({bundle_path.stat().st_size/2**20:.1f} MiB)')
print('Download the zip and preserve the run URL. Do not publish the output.')
